# Лабораторная работа №7. Анализ текста

**Выполнил:** Корнеев Фёдор  
**Группа:** P3120

В работе исследуется классификация русскоязычных твитов по тональности.

# Инструменты для работы с языком

... или зачем нужна предобработка.

Раньше мы смотрели на светлую сторону анализа данных - построение моделей. Теперь попробуем глубже посмотреть на часть про предобработку данных. Задача предобработки особенно актуальна, если мы имеем дело с текстами.

## Задача: классификация твитов по тональности

У нас есть выборка из твитов.
Нам известна эмоциональная окраска каждого твита из выборки: положительная или отрицательная. Задача состоит в построении модели, которая по тексту твита предсказывает его эмоциональную окраску.

Классификацию по тональности используют в рекомендательных системах, чтобы понять, понравилось ли людям кафе, кино, etc.

Скачиваем выборку ([источник](http://study.mokoron.com/)): [положительные](https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/positive.csv), [отрицательные]( https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/negative.csv).

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

data_urls = {
    "positive.csv": (
        "https://raw.githubusercontent.com/Gavroshe/"
        "RuTweetCorp/master/positive.csv"
    ),
    "negative.csv": (
        "https://raw.githubusercontent.com/Gavroshe/"
        "RuTweetCorp/master/negative.csv"
    ),
}

for file_name, url in data_urls.items():
    if not Path(file_name).exists():
        urlretrieve(url, file_name)

In [2]:
import pandas as pd # библиотека для удобной работы с датафреймами
import numpy as np # библиотека для удобной работы со списками и матрицами

# библиотека, где реализованы основные алгоритмы машинного обучения
from sklearn.metrics import *
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [3]:
pd.read_csv('positive.csv', sep=';', header=None).head()

,0,1,2,3,4,5,6,7,8,9,10,11
0,408906692374446080,1386325927,pleease_shut_up,"@first_timee хоть я и школота, но поверь, у на...",1,0,0,0,7569,62,61,0
1,408906692693221377,1386325927,alinakirpicheva,"Да, все-таки он немного похож на него. Но мой ...",1,0,0,0,11825,59,31,2
2,408906695083954177,1386325927,EvgeshaRe,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1,0,1,0,1273,26,27,0
3,408906695356973056,1386325927,ikonnikova_21,"RT @digger2912: ""Кто то в углу сидит и погибае...",1,0,1,0,1549,19,17,0
4,408906761416867842,1386325943,JumpyAlex,@irina_dyshkant Вот что значит страшилка :D\nН...,1,0,0,0,597,16,23,1


In [4]:
pd.read_csv('negative.csv', sep=';', header=None).tail()

,0,1,2,3,4,5,6,7,8,9,10,11
111918,425138243257253888,1390195830,Yanch_96,Но не каждый хочет что то исправлять:( http://...,-1,0,0,0,1138,32,46,0
111919,425138339503943682,1390195853,tkit_on,скучаю так :-( только @taaannyaaa вправляет мо...,-1,0,0,0,4822,38,32,0
111920,425138437684215808,1390195876,ckooker1,"Вот и в школу, в говно это идти уже надо(",-1,0,0,1,165,13,16,0
111921,425138490452344832,1390195889,LisaBeroud,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",-1,0,1,0,2516,187,265,0
111922,425138595251625984,1390195914,sukapavlov,Такси везет меня на работу. Раздумываю приплат...,-1,0,0,0,7778,146,66,5


Откроем файлы и создадим массив из текстов и правильных меток для твитов.
Сначала идут положительные твиты, потом отрицательные.

In [5]:
# Загружаем тексты и присваиваем им известную тональность.
positive = pd.read_csv(
    "positive.csv",
    sep=";",
    usecols=[3],
    names=["text"],
)
positive["label"] = "positive"

negative = pd.read_csv(
    "negative.csv",
    sep=";",
    usecols=[3],
    names=["text"],
)
negative["label"] = "negative"

df = pd.concat([positive, negative], ignore_index=True)
print(f"Всего твитов: {len(df):,}")
df["label"].value_counts()

Всего твитов: 226,834


label
positive    114911
negative    111923
Name: count, dtype: int64

In [6]:
df

,text,label
0,"@first_timee хоть я и школота, но поверь, у на...",positive
1,"Да, все-таки он немного похож на него. Но мой ...",positive
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,positive
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",positive
4,@irina_dyshkant Вот что значит страшилка :D\nН...,positive
...,...,...
226829,Но не каждый хочет что то исправлять:( http://...,negative
226830,скучаю так :-( только @taaannyaaa вправляет мо...,negative
226831,"Вот и в школу, в говно это идти уже надо(",negative
226832,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",negative


Посмотрим на полученные данные:

In [7]:
df.sample(5, random_state=40)

,text,label
15931,RT @Blawar_1337: Теперь у нас с @Wake_UA появи...,positive
59532,с днём рождения зайка*))) ухх погуляем мы сего...,positive
162096,RT @Shumkova0406199: @ann_safina Вов вов вов А...,negative
42002,"Надо выдернуть звуковую дорожку из ""Доктора Ка...",positive
223946,@_hassliebe_ может все таки на этой неделе вер...,negative


Разбиваем данные на обучающую и тестовую выборки с помощью функции ```train_test_split()``` из **sklearn**:


In [8]:
x_train, x_test, y_train, y_test = train_test_split(
    df.text,
    df.label,
    test_size=0.25,
    random_state=42,
    stratify=df.label,
)

print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

(170125,) (56709,) (170125,) (56709,)


In [9]:
y_train[:10]

88343     positive
77885     positive
91702     positive
220625    negative
116547    negative
204560    negative
224108    negative
156680    negative
132493    negative
47399     positive
Name: label, dtype: str

In [10]:
y_train.value_counts()

label
positive    86183
negative    83942
Name: count, dtype: int64

## Baseline: классификация необработанных n-грамм

* Сейчас мы попробуем получить преобразование предложений в численный вектор, с которым может работать стандартный алгоритм машинного обучения, такой как логистическая регрессия.
* Для этого нам понадобится познакомиться с понятием n-gram - самых мелких элементов предложения, с которыми можно работать.
* Подсчитав количество этих n-грам в предложениях, мы получим искомые численные представления.

## Что такое n-граммы:

Самые мелкие структуры языка, с которыми мы работаем, называются **n-граммами**.
У n-граммы есть параметр n - количество слов, которые попадают в такое представление текста.
* Если n = 1 - то мы смотрим на то, сколько раз каждое слово встретилось в тексте. Получаем _униграммы_
* Если n = 2 - то мы смотрим на то, сколько раз каждая пара подряд идущих слов, встретилась в тексте. Получаем _биграммы_

**Функция** для работы с n-граммами реализована в библиотке **nltk** (Natural Language ToolKit), импортируем эту функцию:

In [11]:
from nltk import ngrams

Прежде чем получать n-граммы, нужно разделить предложение на отдельные слова.  Для этого используем метод ```split()```.

In [12]:
sentence = 'Если б мне платили каждый раз'.split()
sentence

['Если', 'б', 'мне', 'платили', 'каждый', 'раз']

Чтобы получить n-грамму для такой последовательности, используем функцию ```ngrams()```.

На вход передается два параметра:
* лист с разделенным на отдельные слова предложением (у нас он хранится в переменной ```sent```);
* параметр n, определяющий, какой тип n-грамм мы хотим получить.


Чтобы полученный объект отобразить, делаем из него ```list```.

In [13]:
list(ngrams(sentence, 1)) # униграммы

[('Если',), ('б',), ('мне',), ('платили',), ('каждый',), ('раз',)]

Аналогично мы можем получить биграммы - для этого заменяем параметр **n** в функции **ngrams** с 1 на 2.

In [14]:
list(ngrams(sentence, 2)) # биграммы

[('Если', 'б'),
 ('б', 'мне'),
 ('мне', 'платили'),
 ('платили', 'каждый'),
 ('каждый', 'раз')]

In [15]:
list(ngrams(sentence, 3)) # триграммы

[('Если', 'б', 'мне'),
 ('б', 'мне', 'платили'),
 ('мне', 'платили', 'каждый'),
 ('платили', 'каждый', 'раз')]

In [16]:
list(ngrams(sentence, 5)) # ... пентаграммы?

[('Если', 'б', 'мне', 'платили', 'каждый'),
 ('б', 'мне', 'платили', 'каждый', 'раз')]

### Векторизаторы

Векторизатор преобразует слово или набор слов в числовой вектор, понятный алгоритму машинного обучения, который привык работать с числовыми табличными данными.

Ниже - пример преобразования слов в двумерных вектор, каждому слову соответствует точка на плоскости.

<a href="https://drive.google.com/uc?id=1ukv-FTj0jeVdcgVlOaNBocUfNuYGGVZg
" target="_blank"><img src="https://drive.google.com/uc?id=1ukv-FTj0jeVdcgVlOaNBocUfNuYGGVZg"
alt="IMAGE ALT TEXT HERE" width="600" border="0" /></a>

На начальном этапе нам будет достаточно тех инструментов, которые уже есть в знакомой нам библиотеке **sklearn**.

In [17]:
from sklearn.linear_model import LogisticRegression # можно заменить на любимый классификатор
from sklearn.feature_extraction.text import CountVectorizer # модель "мешка слов", см. далее

Самый простой способ извлечь признаки из текстовых данных -- векторизаторы: `CountVectorizer` и `TfidfVectorizer`

Объект `CountVectorizer` делает следующую вещь:
* строит для каждого документа (каждой пришедшей ему строки) вектор размерности `n`, где `n` -- количество слов или n-грам во всём корпусе
* заполняет каждый i-тый элемент количеством вхождений слова в данный документ

<a href="https://drive.google.com/uc?id=1ukv-FTj0jeVdcgVlOaNBocUfNuYGGVZg
" target="_blank"><img src="https://drive.google.com/uc?id=1jHmkrGZTMawM46Yzxh243Ur1y5pYKzrl"
alt="IMAGE ALT TEXT HERE" width="600" border="0" /></a>

На рисунке пример векторизации для униграмм, но можно использовать любые n-граммы. Для этого у объекта ```CountVectorizer()``` есть параметр **ngram_range**, который отвечает за то, какие n-граммы мы используем в качестве признаов:<br/>
ngram_range=(1, 1) -- униграммы<br/>
ngram_range=(3, 3) -- триграммы<br/>
ngram_range=(1, 3) -- униграммы, биграммы и триграммы.

<a href="https://drive.google.com/uc?id=1ODNVK0fdLTX4nv6ob55ciUe37d1pio-D" target="_blank"><img src="https://drive.google.com/uc?id=1ODNVK0fdLTX4nv6ob55ciUe37d1pio-D"
alt="IMAGE ALT TEXT HERE" width="800" border="0" /></a>

Инициализируем ```CountVectorizer()```, указав в качестве признаков униграммы:

In [18]:
vectorizer = CountVectorizer(ngram_range=(1,1))

После инициализации _vectorizer_ можно обучить на наших данных.

Для обучения используем обучающую выборку ```x_train```, но в отличие от классификатора мы используем метод ```fit_transform()```: сначала обучаем наш векторизатор, а потом сразу применяем его к нашему набору данных. Это похоже на то, как мы работали с label encoderом и one-hot-encoderом.


In [19]:
vectorized_x_train = vectorizer.fit_transform(x_train)
print("Размер обучающей матрицы:", vectorized_x_train.shape)

Размер обучающей матрицы: (170125, 244014)


In [20]:
vectorized_x_train.shape

(170125, 244014)

Так как результат не зависит от порядка слов в текстах, то говорят, что такая модель представления текстов в виде векторов получается из *гипотезы представления текста как мешка слов*

В vectorizer.vocabulary_ лежит словарь, отображение слов в их индексы:

In [21]:
list(vectorizer.vocabulary_.items())[:10]

[('yaoi_uhahahah', 94355),
 ('ахаха', 102253),
 ('ну', 169630),
 ('да', 123163),
 ('его', 130214),
 ('светка', 207391),
 ('так', 219842),
 ('назвала', 162857),
 ('нему', 166855),
 ('приелось', 193866)]

В нашей выборке 170125 текстов (твитов), в них встречается 243760 разных слов.

In [22]:
vectorized_x_train.shape

(170125, 244014)

Так как теперь у нас есть **численное представление** и набор входных признаков, то мы можем обучить модель логистической регрессии (или любую другую из тех, на которые мы смотрели раньше, например, случайный лес)

In [23]:
baseline_logistic = LogisticRegression(random_state=42, max_iter=1000)
baseline_logistic.fit(vectorized_x_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in 

С тестовыми данными нужно проделать то же самое, что и с данными для обучения: сделать из текстов вектора, которые можно передавать в классификатор для прогноза класса объекта.

У нас уже есть обученный векторизатор ```vectorizer```, поэтому используем метод ```transform()``` (просто применить его), а не ```fit_transform``` (обучить и применить).

In [24]:
vectorized_x_test = vectorizer.transform(x_test)

Как раньше, для получения прогноза у обученного классификатора используем метод ```predict()```.

С помощью функции ```classification_report()```, которая считает сразу несколько метрик качества классификации, посмотрим на то, насколько хорошо мы предсказываем положительную или отрицательную тональность твита .

In [25]:
baseline_predictions = baseline_logistic.predict(vectorized_x_test)
baseline_report = classification_report(
    y_test,
    baseline_predictions,
    output_dict=True,
)
print(classification_report(y_test, baseline_predictions))

              precision    recall  f1-score   support

    negative       0.76      0.77      0.77     27981
    positive       0.77      0.76      0.77     28728

    accuracy                           0.77     56709
   macro avg       0.77      0.77      0.77     56709
weighted avg       0.77      0.77      0.77     56709



## Бонус*: триграммы

Попробуем сделать то же самое, используя в качестве признаков триграммы:

In [26]:
# Триграммы: редкие сочетания отбрасываются для экономии памяти.
vectorizer_3 = CountVectorizer(ngram_range=(3, 3), min_df=2)
vectorized_x_train_3 = vectorizer_3.fit_transform(x_train)

trigram_count_model = LogisticRegression(random_state=42, max_iter=1000)
trigram_count_model.fit(vectorized_x_train_3, y_train)

vectorized_x_test_3 = vectorizer_3.transform(x_test)
trigram_count_predictions = trigram_count_model.predict(vectorized_x_test_3)
print(classification_report(y_test, trigram_count_predictions))

              precision    recall  f1-score   support

    negative       0.73      0.35      0.48     27981
    positive       0.58      0.87      0.70     28728

    accuracy                           0.62     56709
   macro avg       0.66      0.61      0.59     56709
weighted avg       0.66      0.62      0.59     56709



У триграмм словарь намного более разреженный: конкретное сочетание из трёх слов
часто встречается только один раз. Поэтому модель хуже обобщает новые твиты, а
короткие сообщения могут вообще не дать ни одной триграммы. `min_df=2` удаляет
единичные сочетания и одновременно ограничивает расход памяти.

## Бонус**: TF-IDF векторизация

`TfidfVectorizer` делает то же, что и `CountVectorizer`, но в качестве значений выдает **tf-idf** каждого слова.

Как считается tf-idf:

**TF (term frequency)** – относительная частотность слова в документе:
$$ TF(t,d) = \frac{n_{t}}{\sum_k n_{k}} $$

**IDF (inverse document frequency)** – обратная частота документов, в которых есть это слово:
$$ IDF(t, D) = \mbox{log} \frac{|D|}{|{d : t \in d}|} $$

Перемножаем их:
$$TFIDF(t, d, D) = TF(t,d) \times IDF(i, D)$$

Сакральный смысл: если слово часто встречается в одном документе, но в целом по корпусу встречается в небольшом
количестве документов, у него высокий TF-IDF.

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

Действуем аналогично, как с ```CountVectorizer()```:

In [28]:
# TF-IDF для униграмм.
tfidf_vectorizer_1 = TfidfVectorizer(ngram_range=(1, 1), min_df=2)
tfidf_train_1 = tfidf_vectorizer_1.fit_transform(x_train)

tfidf_model_1 = LogisticRegression(random_state=42, max_iter=1000)
tfidf_model_1.fit(tfidf_train_1, y_train)

tfidf_test_1 = tfidf_vectorizer_1.transform(x_test)
tfidf_predictions_1 = tfidf_model_1.predict(tfidf_test_1)
tfidf_report_1 = classification_report(
    y_test,
    tfidf_predictions_1,
    output_dict=True,
)
print(classification_report(y_test, tfidf_predictions_1))

              precision    recall  f1-score   support

    negative       0.77      0.74      0.76     27981
    positive       0.76      0.78      0.77     28728

    accuracy                           0.76     56709
   macro avg       0.76      0.76      0.76     56709
weighted avg       0.76      0.76      0.76     56709



In [29]:
# TF-IDF для пентаграмм.
tfidf_vectorizer_5 = TfidfVectorizer(ngram_range=(5, 5), min_df=2)
tfidf_train_5 = tfidf_vectorizer_5.fit_transform(x_train)

tfidf_model_5 = LogisticRegression(random_state=42, max_iter=1000)
tfidf_model_5.fit(tfidf_train_5, y_train)

tfidf_test_5 = tfidf_vectorizer_5.transform(x_test)
tfidf_predictions_5 = tfidf_model_5.predict(tfidf_test_5)
tfidf_report_5 = classification_report(
    y_test,
    tfidf_predictions_5,
    output_dict=True,
)
print(classification_report(y_test, tfidf_predictions_5))

              precision    recall  f1-score   support

    negative       0.97      0.06      0.11     27981
    positive       0.52      1.00      0.69     28728

    accuracy                           0.54     56709
   macro avg       0.75      0.53      0.40     56709
weighted avg       0.74      0.54      0.40     56709



## Токенизация

Токенизировать - значит, поделить текст на части: слова, ключевые слова, фразы, символы и т.д., иными словами **токены**.

Самый наивный способ токенизировать текст - разделить с помощью функции `split()`. Но `split` упускает очень много всего, например, не отделяет пунктуацию от слов. Кроме этого, есть ещё много менее тривиальных проблем, поэтому лучше использовать готовые токенизаторы.

In [30]:
import nltk # уже знакомая нам библиотека nltk
from nltk.tokenize import word_tokenize # готовый токенизатор библиотеки nltk

Чтобы использовать токенизатор ```word_tokenize```, нужно сначала скачать данные для nltk о пунктуации и стоп-словах. Это просто требование nltk, поэтому, особо не задумываясь, запустите следующую ячейку:  

In [31]:
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

True

Применим токенизацию:

In [32]:
example = 'Но не каждый хочет что-то исправлять:('
word_tokenize(example)

['Но', 'не', 'каждый', 'хочет', 'что-то', 'исправлять', ':', '(']

Если использовать просто ```split()```, то грустный смайлик :( не отделяется от слова "исправлять":

In [33]:
example.split()

['Но', 'не', 'каждый', 'хочет', 'что-то', 'исправлять:(']

В nltk вообще есть довольно много токенизаторов:

In [34]:
from nltk import tokenize
dir(tokenize)[:16]

['BlanklineTokenizer',
 'LegalitySyllableTokenizer',
 'LineTokenizer',
 'MWETokenizer',
 'NLTKWordTokenizer',
 'PunktSentenceTokenizer',
 'PunktTokenizer',
 'RegexpTokenizer',
 'ReppTokenizer',
 'SExprTokenizer',
 'SpaceTokenizer',
 'StanfordSegmenter',
 'SyllableTokenizer',
 'TabTokenizer',
 'TextTilingTokenizer',
 'ToktokTokenizer']

Они умеют выдавать индексы в строке для начала и конца каждого слова-токена:

In [35]:
wh_tok = tokenize.WhitespaceTokenizer()
list(wh_tok.span_tokenize(example))

[(0, 2), (3, 5), (6, 12), (13, 18), (19, 25), (26, 38)]

Некторые токенизаторы ведут себя специфично:

In [36]:
tokenize.TreebankWordTokenizer().tokenize("don't stop me")

['do', "n't", 'stop', 'me']

А некоторые -- вообще не для текста на естественном языке:

In [37]:
tokenize.SExprTokenizer().tokenize("(a (b c)) d e (f)")

['(a (b c))', 'd', 'e', '(f)']

**Правильный токенизатор подбирается исходя из требований задачи!**

## Стоп-слова и пунктуация

**Стоп-слова** - это слова, которые часто встречаются практически в любом тексте и ничего интересного не говорят о конретном документе. Для модели это просто шум. А шум нужно убирать. По аналогичной причине убирают и пунктуацию.

In [38]:
# импортируем стоп-слова из библиотеки nltk
from nltk.corpus import stopwords

# посмотрим на стоп-слова для русского языка
print(stopwords.words('russian'))

['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя', 'впр

*Знаки* пунктуации лучше импортировать из модуля **String**. В нем хранятся различные наборы констант для работы со строками (пунктуация, алфавит и др.).

In [39]:
from string import punctuation
punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

Объединим стоп-слова и знаки пунктуации вместе и запишем в переменную ```noise```:

In [40]:
noise = stopwords.words("russian") + list(punctuation) + ["``", "''"]

Теперь нужно обучать нашу модель с учетом новых знаний про токенизацию и стоп-слова.

Для этого мы можем собрать новый векторизатор, передав ему на вход:
* какие n-граммы нам нужны, параметр **ngram_range**;
* какой токенизатор мы используем, параметр **tokenizer**;
* какие у нас стоп-слова, параметр **stop_words**.

*Напоминание:* мы используем готовый токенизатор ```word_tokenize```, а стоп-слова хранятся в переменной ```noise```

In [41]:
smart_tokenizer = CountVectorizer(
    ngram_range=(1, 1),
    tokenizer=word_tokenize,
    token_pattern=None,
    stop_words=noise,
)

In [42]:
smart_train = smart_tokenizer.fit_transform(x_train)

smart_model = LogisticRegression(random_state=42, max_iter=1000)
smart_model.fit(smart_train, y_train)

smart_test = smart_tokenizer.transform(x_test)
smart_predictions = smart_model.predict(smart_test)
print(classification_report(y_test, smart_predictions))

              precision    recall  f1-score   support

    negative       0.76      0.80      0.78     27981
    positive       0.80      0.75      0.77     28728

    accuracy                           0.78     56709
   macro avg       0.78      0.78      0.78     56709
weighted avg       0.78      0.78      0.78     56709



Получилось лучше: accuracy выше, а также заметно подрос recall у негативного класса.

Что ещё можно сделать?

## Бонус*: Лемматизация

**Лемматизация** – это сведение разных форм одного слова к начальной форме – **лемме**. Почему это хорошо?
* Во-первых, естественно рассматривать как отдельный признак каждое *слово*, а не каждую его отдельную форму.
* Во-вторых, некоторые стоп-слова стоят только в начальной форме, и без лематизации выкидываем мы только её.

Для русского есть хороших лемматизатор pymorphy.

### [Pymorphy](http://pymorphy2.readthedocs.io/en/latest/)
Это модуль на питоне, довольно быстрый и с кучей функций.

In [43]:
# В Colab пакет устанавливается этой командой; локально он уже установлен.
%pip install -q pymorphy3


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/infinitrator/infinitrator.github.io/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


В pymorphy2 для морфологического анализа слов есть ```MorphAnalyzer()```:

In [44]:
from pymorphy3 import MorphAnalyzer

pymorphy3_analyzer = MorphAnalyzer()

pymorphy3 работает с отдельными словами. Если дать ему на вход предложение - он его просто не лемматизирует, т.к. не понимает:

In [45]:
sent = ['Если', 'б', 'мне', 'платили', 'каждый', 'раз']
sent

['Если', 'б', 'мне', 'платили', 'каждый', 'раз']

Лемматизируем слово "платили" из предложения ```sent``` с помощью метода ```parse()```:

In [46]:
ana = pymorphy3_analyzer.parse(sent[3])
ana[:2]

[Parse(word='платили', tag=OpencorporaTag('VERB,impf,tran plur,past,indc'), normal_form='платить', score=1.0, methods_stack=((DictionaryAnalyzer(), 'платили', 2471, 10),))]

Выведем его нормальную форму:

In [47]:
ana[0].normal_form

'платить'

## О важности эксплоративного анализа

Но иногда пунктуация бывает и не шумом - главное отталкиваться от задачи. Что будет если вообще не убирать пунктуацию?

In [48]:
# Оставляем пунктуацию и смайлики как отдельные токены.
punctuation_vectorizer = CountVectorizer(
    ngram_range=(1, 1),
    tokenizer=word_tokenize,
    token_pattern=None,
    stop_words=None,
)
punctuation_train = punctuation_vectorizer.fit_transform(x_train)

punctuation_model = LogisticRegression(random_state=42, max_iter=1000)
punctuation_model.fit(punctuation_train, y_train)

punctuation_test = punctuation_vectorizer.transform(x_test)
punctuation_predictions = punctuation_model.predict(punctuation_test)
print(classification_report(y_test, punctuation_predictions))

              precision    recall  f1-score   support

    negative       1.00      1.00      1.00     27981
    positive       1.00      1.00      1.00     28728

    accuracy                           1.00     56709
   macro avg       1.00      1.00      1.00     56709
weighted avg       1.00      1.00      1.00     56709



Качество стало почти идеальным не из-за глубокого понимания текста. Корпус был
собран по эмоциональным смайликам, поэтому `:)`, `:(`, `:D` и похожие символы
одновременно участвовали в назначении метки и остались в тексте. Это **утечка
целевого признака**: модель восстанавливает правило разметки. Для реального
анализа качества такие смайлики следовало бы удалить до разделения выборки.

Посмотрим, как один из супер-значительных токенов справится с классификацией безо всякого машинного обучения:

In [49]:
def emoticon_classifier(text: str) -> str:
    # Классификация только по эмоциональным смайликам.
    positive_markers = (":)", ":D", ";)", "=)")
    return "positive" if any(marker in text for marker in positive_markers) else "negative"


emoticon_predictions = x_test.apply(emoticon_classifier)
print(classification_report(y_test, emoticon_predictions))

              precision    recall  f1-score   support

    negative       0.61      1.00      0.76     27981
    positive       1.00      0.38      0.55     28728

    accuracy                           0.69     56709
   macro avg       0.81      0.69      0.66     56709
weighted avg       0.81      0.69      0.65     56709



## Символьные n-граммы

Теперь в качестве фичей используем, например, униграммы символов. Для этого необходимо установить в ```CountVectorizer()``` параметр ```analyzer = 'char'```, то есть анализировать символы.

In [50]:
char_vectorizer = CountVectorizer(analyzer="char", ngram_range=(1, 1))
char_train = char_vectorizer.fit_transform(x_train)

char_model = LogisticRegression(random_state=42, max_iter=1000)
char_model.fit(char_train, y_train)

char_test = char_vectorizer.transform(x_test)
char_predictions = char_model.predict(char_test)
print(classification_report(y_test, char_predictions))

              precision    recall  f1-score   support

    negative       1.00      0.99      1.00     27981
    positive       0.99      1.00      1.00     28728

    accuracy                           1.00     56709
   macro avg       1.00      1.00      1.00     56709
weighted avg       1.00      1.00      1.00     56709



Из предыдущего раздела уже понятно, почему на этих данных точность равна 1.

Символьные n-граммы используются, например, для задачи определения языка. Ещё одна замечательная особенность признаков-символов - для них не нужна токенизация и лемматизация, можно использовать такой подход для языков, у которых нет готовых анализаторов.

# Самостоятельная работа

1. Изучите материал, представленный в борде.
2. Выполните все ячейки и получите результаты.
3. Приведите результаты таблицы classification_report в под этим заданием для модели LogisticRegression
4. Примените 2 альтернативных использованному алгоритму для решения задачи классификации (для примера XGBClassifier и еще какой-то один) и получите результаты в таблице classification_report
5. Для XGBClassifier вам потребуется задать параметры
```learning_rate=0.1, n_estimators=1000, max_depth=5, min_child_weight=3, gamma=0.2, subsample=0.6, colsample_bytree=1.0, objective='binary:logistic', nthread=4, scale_pos_weight=1, seed=27```

6. В разделе TF-IDF векторизация по аналогии с униграммами и пентаграммами вычислите classification_report для биграмм, триграмм опубликуйте результаты в отчете и укажите изменилась ли точность f1-score при их использовании по сравнению с униграммами и пентаграммами.


## Самостоятельная работа

### Сравнение TF-IDF для разных n-грамм

Для биграмм и триграмм повторяется тот же эксперимент, что для униграмм и
пентаграмм. Во всех случаях используется одинаковое разбиение и
`LogisticRegression`, поэтому изменение метрик связано с представлением текста.
Редкие n-граммы с частотой меньше двух удаляются для ограничения памяти.


In [51]:

tfidf_reports = {
    1: tfidf_report_1,
    5: tfidf_report_5,
}

for ngram_size in (2, 3):
    current_vectorizer = TfidfVectorizer(
        ngram_range=(ngram_size, ngram_size),
        min_df=2,
    )
    current_train = current_vectorizer.fit_transform(x_train)
    current_model = LogisticRegression(random_state=42, max_iter=1000)
    current_model.fit(current_train, y_train)
    current_test = current_vectorizer.transform(x_test)
    current_predictions = current_model.predict(current_test)
    tfidf_reports[ngram_size] = classification_report(
        y_test,
        current_predictions,
        output_dict=True,
    )
    print(f"TF-IDF, {ngram_size}-граммы")
    print(classification_report(y_test, current_predictions))


TF-IDF, 2-граммы
              precision    recall  f1-score   support

    negative       0.72      0.65      0.69     27981
    positive       0.69      0.76      0.72     28728

    accuracy                           0.71     56709
   macro avg       0.71      0.71      0.71     56709
weighted avg       0.71      0.71      0.71     56709



TF-IDF, 3-граммы
              precision    recall  f1-score   support

    negative       0.73      0.36      0.48     27981
    positive       0.58      0.87      0.70     28728

    accuracy                           0.62     56709
   macro avg       0.66      0.62      0.59     56709
weighted avg       0.66      0.62      0.59     56709



In [52]:

tfidf_comparison = pd.DataFrame(
    {
        f"{ngram_size}-граммы": {
            "accuracy": report["accuracy"],
            "precision weighted": report["weighted avg"]["precision"],
            "recall weighted": report["weighted avg"]["recall"],
            "f1 weighted": report["weighted avg"]["f1-score"],
        }
        for ngram_size, report in sorted(tfidf_reports.items())
    }
).T
tfidf_comparison.round(4)


,accuracy,precision weighted,recall weighted,f1 weighted
1-граммы,0.7621,0.7623,0.7621,0.7620
2-граммы,0.7064,0.7080,0.7064,0.7056
3-граммы,0.6191,0.6562,0.6191,0.5922
5-граммы,0.5356,0.7431,0.5356,0.4036



Чем больше `n`, тем точнее выражение хранит контекст, но тем реже оно повторяется
в новых сообщениях. Поэтому на коротких твитах униграммы обычно обобщают лучше,
а чистые пентаграммы дают много пустых векторов. Точные выводы ниже основаны на
полученной таблице, а не на предположении.



### Два альтернативных алгоритма

Для честного сравнения один TF-IDF-векторизатор обучается только на обучающей
части. Проверяются `MultinomialNB` и заданный в условии `XGBClassifier`.
XGBoost принимает бинарные метки: negative = 0, positive = 1.


In [53]:

from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier

alternative_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=50_000,
    sublinear_tf=True,
)
alternative_train = alternative_vectorizer.fit_transform(x_train)
alternative_test = alternative_vectorizer.transform(x_test)

naive_bayes_model = MultinomialNB(alpha=0.5)
naive_bayes_model.fit(alternative_train, y_train)
naive_bayes_predictions = naive_bayes_model.predict(alternative_test)

print("MultinomialNB")
print(classification_report(y_test, naive_bayes_predictions))


MultinomialNB
              precision    recall  f1-score   support

    negative       0.75      0.75      0.75     27981
    positive       0.76      0.76      0.76     28728

    accuracy                           0.76     56709
   macro avg       0.76      0.76      0.76     56709
weighted avg       0.76      0.76      0.76     56709



In [54]:

label_mapping = {"negative": 0, "positive": 1}
inverse_label_mapping = {0: "negative", 1: "positive"}
y_train_binary = y_train.map(label_mapping)

xgb_model = XGBClassifier(
    learning_rate=0.1,
    n_estimators=1000,
    max_depth=5,
    min_child_weight=3,
    gamma=0.2,
    subsample=0.6,
    colsample_bytree=1.0,
    objective="binary:logistic",
    nthread=4,
    scale_pos_weight=1,
    seed=27,
    tree_method="hist",
    eval_metric="logloss",
)
xgb_model.fit(alternative_train, y_train_binary)
xgb_binary_predictions = xgb_model.predict(alternative_test)
xgb_predictions = pd.Series(xgb_binary_predictions).map(inverse_label_mapping)

print("XGBClassifier")
print(classification_report(y_test.to_numpy(), xgb_predictions))


XGBClassifier
              precision    recall  f1-score   support

    negative       0.75      0.69      0.72     27981
    positive       0.72      0.77      0.74     28728

    accuracy                           0.73     56709
   macro avg       0.73      0.73      0.73     56709
weighted avg       0.73      0.73      0.73     56709



In [55]:

model_reports = {
    "LogisticRegression (Count)": baseline_report,
    "MultinomialNB (TF-IDF 1-2)": classification_report(
        y_test,
        naive_bayes_predictions,
        output_dict=True,
    ),
    "XGBClassifier (TF-IDF 1-2)": classification_report(
        y_test.to_numpy(),
        xgb_predictions,
        output_dict=True,
    ),
}

model_comparison = pd.DataFrame(
    {
        model_name: {
            "accuracy": report["accuracy"],
            "precision weighted": report["weighted avg"]["precision"],
            "recall weighted": report["weighted avg"]["recall"],
            "f1 weighted": report["weighted avg"]["f1-score"],
        }
        for model_name, report in model_reports.items()
    }
).T.sort_values("f1 weighted", ascending=False)
model_comparison.round(4)


,accuracy,precision weighted,recall weighted,f1 weighted
LogisticRegression (Count),0.7665,0.7666,0.7665,0.7665
MultinomialNB (TF-IDF 1-2),0.7564,0.7563,0.7564,0.7563
XGBClassifier (TF-IDF 1-2),0.7313,0.7322,0.7313,0.7309



### Итог

Выполнены все упражнения исходного борда, включая токенизацию, удаление шума,
лемматизацию, проверку пунктуации и символьные признаки. Самостоятельно сравнены
четыре размера TF-IDF n-грамм и три классификатора. Почти идеальные результаты
при сохранении пунктуации нельзя считать честным качеством анализа тональности:
они объясняются утечкой смайликов, по которым исходный корпус был размечен.
